# Assignment Quantile Sweep

Sweep one graph-construction parameter on a 2,000-cell pancreas subset and compare the paired batch/cell-type embeddings in a grid.

In [1]:
def _coerce_sweep_value(value, template):
    if isinstance(template, bool) or template is None:
        return value
    if isinstance(template, int):
        return int(value)
    if isinstance(template, float):
        return float(value)
    return value


def run_parameter_sweep(adata, *, base_graph_params, sweep_parameter, sweep_values):
    results = []
    for sweep_value in sweep_values:
        adata_sweep = adata.copy()
        graph_params = base_graph_params.copy()
        graph_params[sweep_parameter] = _coerce_sweep_value(sweep_value, base_graph_params[sweep_parameter])
        adata_sweep.obsm["X_scalp"] = estimator.embed(
            adata_sweep,
            **graph_params,
            embedding_random_state=0,
        )
        results.append((graph_params[sweep_parameter], adata_sweep))
    return results


def _format_sweep_value(value):
    return f"{value:.2f}" if isinstance(value, float) else str(value)


def plot_parameter_sweep(results, *, sweep_parameter):
    n_rows = len(results)
    fig, axes = plt.subplots(n_rows, 2, figsize=(15, 4 * n_rows), constrained_layout=True)
    axes = np.asarray(axes).reshape(n_rows, 2)

    for row, (sweep_value, adata_sweep) in enumerate(results):
        estimator.plot(
            adata_sweep,
            embedding_key="X_scalp",
            axes=axes[row],
            s=8,
            alpha=0.85,
            random_state=0,
            legend_markerscale=3.5,
        )
        formatted_value = _format_sweep_value(sweep_value)
        axes[row, 0].set_title(f"{sweep_parameter}={formatted_value} by {batch_key}")
        axes[row, 1].set_title(f"{sweep_parameter}={formatted_value} by {label_key}")
    return fig


In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from scalp_lite import ScalpEstimator


# Resolve paths whether Jupyter starts in the project root or in notebooks/.
project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists())
# Pancreas benchmark with real experimental studies/platforms as batches.
input_path = project_root / "data" / "pancreas_normalized.h5ad"
# The study column identifies real experimental batches to align.
batch_key = "study"
# The cell_type column identifies biological labels used for visual inspection.
label_key = "cell_type"
# Keep the sweep small enough for repeated graph construction and UMAP embedding.
max_cells = 3000
estimator = ScalpEstimator(batch_key=batch_key, label_key=label_key, random_state=0)
input_path, batch_key, label_key

(PosixPath('/Users/fabriziocosta/Resilio Sync/Sync/Projects/ACTIVE/scalp-lite/data/pancreas_normalized.h5ad'),
 'study',
 'cell_type')

In [3]:
preprocess_params = {
    # Use the already normalized pancreas matrix without another library normalization pass.
    "normalize": False,
    # Variance selection is robust for this pre-normalized development dataset.
    "hvg_flavor": "variance",
    # Keep exactly 2,000 cells, stratified by batch inside ScalpEstimator.preprocess.
    "max_cells": max_cells,
    # Do not drop genes from this already curated benchmark before HVG selection.
    "min_gene_counts": 0,
    # The dataset has real batches, so no artificial split is needed.
    "create_artificial_batch": False,
}

adata = estimator.to_data(input_path)
adata = estimator.preprocess(adata, **preprocess_params)
adata

AnnData object with n_obs × n_vars = 3000 × 1000
    obs: 'batch', 'study', 'cell_type', 'size_factors'
    obsm: 'X_pca'

In [4]:
base_graph_params = {
    # Number of local neighbors considered inside each batch; integer >= 1.
    "n_neighbors": 15,
    # Fraction of graph mass reserved for within-batch neighborhood structure; float from 0 to 1.
    "intra_fraction": 0.5,
    # Number of repeated Hungarian assignment rounds between each batch pair; integer >= 0.
    "n_inter_edges": 4,
    # Distance metric passed to sklearn.metrics.pairwise_distances; common options include
    # "euclidean", "cosine", "manhattan", and any metric accepted by sklearn/scipy.
    "metric": "euclidean",
    # Cross-batch assignment filter; options are None to keep all assignments, or a float in (0, 1].
    "assignment_quantile": 0.9,
    # Hubness correction before assignment and neighbor selection; options are "csls" or "none".
    "hubness_correction": "csls",
    # Number of local distances used for CSLS scale estimation; integer >= 1.
    "hubness_k": 10,
    # Edge weights passed to UMAP/spectral embedding; options are "binary" or "distance".
    "edge_weighting": "binary",
    # Within-batch neighbor rule; True keeps only reciprocal neighbors, False keeps the union.
    "mutual_neighbors": False,
    # Within-batch neighbor scoring; options are "rank" or "distance".
    "neighbor_mode": "distance",
    # Return an undirected graph for downstream spectral/UMAP embedding; options are True or False.
    "symmetrize": True,
}

base_graph_params


{'n_neighbors': 15,
 'intra_fraction': 0.5,
 'n_inter_edges': 4,
 'metric': 'euclidean',
 'assignment_quantile': 0.9,
 'hubness_correction': 'csls',
 'hubness_k': 10,
 'edge_weighting': 'binary',
 'mutual_neighbors': False,
 'neighbor_mode': 'distance',
 'symmetrize': True}

## Assignment Quantile Sweep


In [ ]:
sweep_parameter = "assignment_quantile"
sweep_values = np.linspace(0.01, 0.99, 7)

assignment_quantile_results = run_parameter_sweep(
    adata,
    base_graph_params=base_graph_params,
    sweep_parameter=sweep_parameter,
    sweep_values=sweep_values,
)

[(round(q, 3), a.obsm["X_scalp"].shape) for q, a in assignment_quantile_results]


In [ ]:
_ = plot_parameter_sweep(assignment_quantile_results, sweep_parameter="assignment_quantile")

## Inter-Batch Edge Sweep


In [ ]:
sweep_parameter = "n_inter_edges"
sweep_values = np.linspace(1, 10, 6).astype(int)

n_inter_edges_results = run_parameter_sweep(
    adata,
    base_graph_params=base_graph_params,
    sweep_parameter=sweep_parameter,
    sweep_values=sweep_values,
)

[(round(q, 3), a.obsm["X_scalp"].shape) for q, a in n_inter_edges_results]


In [ ]:
_ = plot_parameter_sweep(n_inter_edges_results, sweep_parameter="n_inter_edges")